# Disclosure-time signal research

**Synthetic demonstration only.** No real performance or predictive edge is claimed.

Research question: do disclosed ownership changes precede subsequent returns? The baseline is fixed in advance and evaluated sequentially; it is not a fitted forecasting model.

In [1]:
from pathlib import Path
import pandas as pd
from crowding.storage import load_dataset
from crowding.backtesting.engine import walk_forward, Config
from crowding.backtesting.attribution import sector_attribution
from crowding.normalization.holdings import snapshot
from crowding.analytics.crowding import crowding_score

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
d = load_dataset(root / "data/demo")
assert d["manifest"]["kind"] == "synthetic"
d["manifest"]

{'kind': 'synthetic',
 'seed': 441,
 'license': 'CC0-1.0',
 'description': 'Generated demonstration only; no empirical investment evidence.',
 'price_adjustment': 'Synthetic total-return index',
 'benchmark': 'Equal-weight synthetic universe',
 'generator_version': 1}

## Audit the information boundary
Actual filing dates govern availability. Reporting dates only identify portfolio periods. The following assertion fails if a future disclosure enters a test fold.

In [2]:
result = walk_forward(d["filings"], d["positions"], d["prices"])
folds = result["folds"].dropna(subset=["latest_filing"])
assert (folds.latest_filing < folds.test_start).all()
result["folds"].tail()

,train_start,train_end,test_start,latest_filing,eligible_securities,held_securities
61,2020-01-01,2025-07-31,2025-08-01,2025-07-30,30,5
62,2020-01-01,2025-08-31,2025-09-01,2025-08-16,30,5
63,2020-01-01,2025-09-30,2025-10-01,2025-08-16,30,5
64,2020-01-01,2025-11-02,2025-11-03,2025-11-02,30,5
65,2020-01-01,2025-11-30,2025-12-01,2025-11-15,30,5


## Cost sensitivity
Zero, 15 and 50 bps per traded dollar. These are illustrative assumptions, not calibrated liquidity costs. Costs use drifted holdings and include initial entry.

In [3]:
runs = {
    bps: walk_forward(d["filings"], d["positions"], d["prices"], Config(cost_bps=bps)) for bps in [0, 15, 50]
}
summary = pd.DataFrame({bps: r["metrics"] for bps, r in runs.items()}).T
assert summary.loc[0, "total_return"] >= summary.loc[15, "total_return"] >= summary.loc[50, "total_return"]
summary

,total_return,annualized_return,volatility,sharpe,max_drawdown
0,0.389149,0.059377,0.167629,0.427879,-0.304045
15,0.263876,0.041952,0.167586,0.328964,-0.320884
50,0.012915,0.002254,0.168330,0.097479,-0.372657


## Forward 1-, 3- and 6-month rank IC
Overlapping horizons are dependent. Mean IC alone does not establish significance. No p-values or trading-edge claims are made.

In [4]:
result["ic"].groupby("months").agg(mean_ic=("ic", "mean"), valid_folds=("ic", "count"))

,mean_ic,valid_folds
months,,
1,-0.003645,64
3,0.027402,62
6,0.016129,59


In [5]:
sector_attribution(result["contributions"], d["mappings"])

,sector,gross_contribution_sum,asset_days
0,Consumer,-0.001142,1310
1,Energy,0.015273,1084
2,Financials,-0.003059,1368
3,Health Care,0.283159,1238
4,Industrials,0.106050,891
5,Technology,0.008436,1284


## Inspect score decomposition

In [6]:
as_of = pd.Timestamp("2025-11-15")
current = snapshot(d["filings"], d["positions"], as_of)
previous = snapshot(d["filings"], d["positions"], as_of - pd.DateOffset(months=3))
crowding_score(current, previous).head(10)

,security_id,breadth,value,holder_hhi,ownership_change,overlap,score,breadth_rank,value_rank,holder_hhi_rank,ownership_change_rank,overlap_rank
22,DEMO00022,7,9.472823e+08,0.149335,0.166667,0.463496,80.862069,0.982759,1.000000,0.000000,0.844828,0.965517
17,DEMO00017,7,5.699427e+08,0.167746,0.333333,0.371089,78.965517,0.982759,0.931034,0.034483,1.000000,0.689655
29,DEMO00029,5,6.346699e+08,0.247006,0.083333,0.484652,70.862069,0.568966,0.965517,0.379310,0.689655,1.000000
5,DEMO00005,6,4.888382e+08,0.215425,0.166667,0.333001,70.000000,0.844828,0.896552,0.206897,0.844828,0.448276
1,DEMO00001,5,3.210614e+08,0.234254,0.166667,0.418852,64.827586,0.568966,0.689655,0.310345,0.844828,0.827586
25,DEMO00025,5,1.722495e+08,0.306907,0.166667,0.442527,64.310345,0.568966,0.275862,0.724138,0.844828,0.931034
27,DEMO00027,5,3.217931e+08,0.209507,0.166667,0.414400,63.620690,0.568966,0.758621,0.172414,0.844828,0.793103
7,DEMO00007,5,2.251050e+08,0.223597,0.250000,0.397024,60.344828,0.568966,0.448276,0.275862,0.965517,0.724138
16,DEMO00016,5,2.883196e+08,0.283790,-0.083333,0.437710,58.620690,0.568966,0.620690,0.586207,0.344828,0.896552
11,DEMO00011,5,3.217090e+08,0.264313,-0.083333,0.347790,54.482759,0.568966,0.724138,0.517241,0.344828,0.551724


## Interpretation and next empirical study
The output validates plumbing, not economic hypotheses. For real research, freeze a manager universe using information available at each date; load historical identifier mappings and licensed total-return prices including delistings; audit amendments and missing filings; preregister parameters and reserve an untouched holdout. If fitting models, purge overlapping labels and use training-only transforms. See docs/METHODOLOGY.md for formulas and exclusions.